In [ ]:
# UAC Predictive Forecasting of Care Load & Placement Demand
# Colab training notebook
# This notebook cleans the HHS UAC dataset, builds forecasting models,
# evaluates them using a time-based split, and saves artifacts for Streamlit.


In [ ]:
# 1. Install / import libraries
!pip -q install pandas numpy matplotlib seaborn scikit-learn statsmodels joblib plotly

import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")
np.random.seed(42)


In [ ]:
# 2. Upload the CSV
from google.colab import files

uploaded = files.upload()
FILE_NAME = list(uploaded.keys())[0]
print("Using:", FILE_NAME)


In [ ]:
# 3. Load and inspect the dataset
raw = pd.read_csv(FILE_NAME)

print("Original shape:", raw.shape)
display(raw.head())
print("\nColumns:")
print(raw.columns.tolist())
print("\nMissing values:")
display(raw.isna().sum())


In [ ]:
# 4. Clean the dataset
# The supplied file contains trailing completely blank rows.
df = raw.dropna(how="all").copy()

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

numeric_cols = [
    "Children apprehended and placed in CBP custody*",
    "Children in CBP custody",
    "Children transferred out of CBP custody",
    "Children in HHS Care",
    "Children discharged from HHS Care"
]

# Remove commas such as "2,484" and convert to numbers.
for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

df = df.dropna(subset=["Date"]).sort_values("Date")
df = df.drop_duplicates(subset="Date")

print("After cleaning:", df.shape)
display(df.head())
display(df.describe())


In [ ]:
# 5. Convert irregular reporting dates to daily data
# The source has gaps between reporting dates, so we create a daily index.
df = df.set_index("Date")

daily = df.asfreq("D")

# Time interpolation is used only for the missing dates created by reindexing.
# Original observations remain unchanged.
daily[numeric_cols] = daily[numeric_cols].interpolate(method="time")
daily[numeric_cols] = daily[numeric_cols].ffill().bfill()

print("Daily dataset:", daily.shape)
print("Date range:", daily.index.min().date(), "to", daily.index.max().date())
display(daily.head())


In [ ]:
# 6. EDA
plt.figure(figsize=(14, 5))
plt.plot(daily.index, daily["Children in HHS Care"])
plt.title("Children in HHS Care Over Time")
plt.xlabel("Date")
plt.ylabel("Children")
plt.grid(alpha=0.25)
plt.show()

plt.figure(figsize=(14, 5))
plt.plot(daily.index, daily["Children discharged from HHS Care"], label="Discharged")
plt.plot(daily.index, daily["Children transferred out of CBP custody"], label="Transferred")
plt.title("Daily Flow Indicators")
plt.xlabel("Date")
plt.ylabel("Children")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


In [ ]:
# 7. Feature engineering
def make_features(data, target):
    X = pd.DataFrame(index=data.index)

    # Historical lag values
    for lag in [1, 2, 3, 7, 14, 21, 28]:
        X[f"lag_{lag}"] = data[target].shift(lag)

    # Rolling statistics use shift(1) so the current target is never used.
    for window in [7, 14, 28]:
        shifted = data[target].shift(1)
        X[f"roll_mean_{window}"] = shifted.rolling(window).mean()
        X[f"roll_std_{window}"] = shifted.rolling(window).std()

    # Calendar features
    X["day_of_week"] = X.index.dayofweek
    X["month"] = X.index.month
    X["day_of_year"] = X.index.dayofyear
    X["week_of_year"] = X.index.isocalendar().week.astype(int).values

    return X

def make_feature_row(history, date):
    s = pd.Series(history, index=pd.to_datetime(history.index)).sort_index()
    row = {}

    for lag in [1, 2, 3, 7, 14, 21, 28]:
        row[f"lag_{lag}"] = float(s.iloc[-lag])

    for window in [7, 14, 28]:
        vals = s.iloc[-window:]
        row[f"roll_mean_{window}"] = float(vals.mean())
        row[f"roll_std_{window}"] = float(vals.std()) if len(vals) > 1 else 0.0

    row["day_of_week"] = date.dayofweek
    row["month"] = date.month
    row["day_of_year"] = date.dayofyear
    row["week_of_year"] = int(date.isocalendar().week)

    return pd.DataFrame([row], index=[date])

def recursive_rf_forecast(model, history, horizon):
    history = history.copy()
    future = []

    for _ in range(horizon):
        next_date = history.index[-1] + pd.Timedelta(days=1)
        X_next = make_feature_row(history, next_date)
        pred = float(model.predict(X_next)[0])
        pred = max(0, pred)
        future.append((next_date, pred))
        history.loc[next_date] = pred

    return pd.Series(dict(future), name="forecast")

def mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.maximum(np.abs(y_true), 1e-8)
    return np.mean(np.abs((y_true - y_pred) / denom)) * 100

def metric_row(name, actual, pred):
    return {
        "Model": name,
        "MAE": mean_absolute_error(actual, pred),
        "RMSE": np.sqrt(mean_squared_error(actual, pred)),
        "MAPE (%)": mape(actual, pred)
    }


In [ ]:
# 8. Time-based train/test split
TARGETS = {
    "care_load": "Children in HHS Care",
    "discharge": "Children discharged from HHS Care",
    "transfer": "Children transferred out of CBP custody"
}

feature_sets = {}
target_sets = {}

for key, target in TARGETS.items():
    X = make_features(daily, target)
    y = daily[target]
    valid = X.notna().all(axis=1)

    feature_sets[key] = X.loc[valid]
    target_sets[key] = y.loc[valid]

split_ratio = 0.80
splits = {}

for key in TARGETS:
    X = feature_sets[key]
    y = target_sets[key]
    split = int(len(X) * split_ratio)
    splits[key] = (X.iloc[:split], X.iloc[split:], y.iloc[:split], y.iloc[split:])

print("80/20 chronological split created.")


In [ ]:
# 9. Train and compare Random Forest + Naive + SARIMA
comparison_rows = []
models_final = {}
validation_residuals = {}

for key, target in TARGETS.items():
    X_train, X_test, y_train, y_test = splits[key]

    # Random Forest
    rf = RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)

    # Naive persistence: tomorrow ~= today
    naive_pred = y_test.shift(0).copy()
    naive_pred[:] = y_train.iloc[-1]
    # Better rolling one-step naive benchmark
    full_y = pd.concat([y_train, y_test])
    naive_pred = full_y.shift(1).loc[y_test.index]

    # SARIMA
    sarima = SARIMAX(
        y_train,
        order=(1, 1, 1),
        seasonal_order=(1, 1, 1, 7),
        enforce_stationarity=False,
        enforce_invertibility=False
    ).fit(disp=False)

    sarima_pred = sarima.get_forecast(steps=len(y_test)).predicted_mean

    comparison_rows += [
        metric_row(f"{key} - Random Forest", y_test, rf_pred),
        metric_row(f"{key} - Naive", y_test, naive_pred),
        metric_row(f"{key} - SARIMA", y_test, sarima_pred)
    ]

    validation_residuals[key] = {
        "Random Forest": (y_test.values - rf_pred).tolist(),
        "Naive": (y_test.values - naive_pred.values).tolist(),
        "SARIMA": (y_test.values - sarima_pred.values).tolist()
    }

    models_final[key] = {}

    # Refit Random Forest on all available feature rows for production forecasting.
    rf_final = RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
    rf_final.fit(feature_sets[key], target_sets[key])
    models_final[key]["Random Forest"] = rf_final

    # Fit SARIMA on the complete daily target series.
    sarima_final = SARIMAX(
        daily[target],
        order=(1, 1, 1),
        seasonal_order=(1, 1, 1, 7),
        enforce_stationarity=False,
        enforce_invertibility=False
    ).fit(disp=False)
    models_final[key]["SARIMA"] = sarima_final

    models_final[key]["Naive"] = None

comparison = pd.DataFrame(comparison_rows)
display(comparison.sort_values(["Model"]))


In [ ]:
# 10. Plot the care-load validation comparison
care_comparison = comparison[comparison["Model"].str.startswith("care_load")].copy()
display(care_comparison)

key = "care_load"
X_train, X_test, y_train, y_test = splits[key]
rf_check = RandomForestRegressor(
    n_estimators=300, max_depth=12, min_samples_leaf=2,
    random_state=42, n_jobs=-1
)
rf_check.fit(X_train, y_train)
pred = rf_check.predict(X_test)

plt.figure(figsize=(14, 5))
plt.plot(y_test.index, y_test.values, label="Actual")
plt.plot(y_test.index, pred, label="Random Forest")
plt.title("Care Load: Validation Forecast")
plt.xlabel("Date")
plt.ylabel("Children in HHS Care")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


In [ ]:
# 11. Save artifacts for Streamlit
ARTIFACT_DIR = "artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# Save daily cleaned data
daily.to_csv(f"{ARTIFACT_DIR}/daily_data.csv")

# Save trained models
joblib.dump(models_final, f"{ARTIFACT_DIR}/models.pkl")

# Save validation residuals for approximate forecast intervals
joblib.dump(validation_residuals, f"{ARTIFACT_DIR}/validation_residuals.pkl")

# Save model comparison table
comparison.to_csv(f"{ARTIFACT_DIR}/model_comparison.csv", index=False)

metadata = {
    "last_date": str(daily.index.max().date()),
    "targets": TARGETS,
    "frequency": "D",
    "random_seed": 42
}
joblib.dump(metadata, f"{ARTIFACT_DIR}/metadata.pkl")

print("Saved:")
for root, dirs, files_ in os.walk(ARTIFACT_DIR):
    for f in files_:
        print(os.path.join(root, f))


In [ ]:
# 12. Optional: download the artifacts
!zip -r uac_streamlit_artifacts.zip artifacts
from google.colab import files
files.download("uac_streamlit_artifacts.zip")
